# SBD — Análisis de captaciones junio 2026

Notebook de análisis del CSV `captaciones_junio_2026.csv` usando **solo pandas y matplotlib**.

Preguntas resueltas:
1. ¿Rangos de edad que mas viajan?.
2. ¿Se conectan mas PCs o mas moviles?.
3. Países de origen más frecuentes.
4. ¿Que idioma ha sido el mas hablado?.

El CSV contiene ~500.000 filas con errores intencionales (nulos, `???`, `edad_XX`, `ESP`), por eso se aplica una limpieza mínima antes de tabular.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# %matplotlib inline

print(f"pandas {pd.__version__}")
import matplotlib
print(f"matplotlib {matplotlib.__version__}")

In [ ]:
# Localizar el CSV tanto si el notebook se abre desde SBD/ como desde la raíz del repo.
candidatos = [Path("../BDA/ImporCsvPandas/captaciones_junio_2026.csv"), Path("BDA/ImporCsvPandas/captaciones_junio_2026.csv")]
CSV = next((p for p in candidatos if p.exists()), candidatos[0])
print(f"CSV: {CSV.resolve()}")

df_raw = pd.read_csv(CSV, encoding="utf-8-sig")
print(f"Filas: {len(df_raw):,} | Columnas: {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# Limpieza mínima (solo pandas): normalizar texto, descartar nulos y valores rotos.
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
for c in ["OS del sistema", "Idioma del OS", "Pais de origen"]:
    df[c] = df[c].astype("string").str.strip()
df["edad"] = pd.to_numeric(df["Rango de edad"], errors="coerce")

OS_VALIDOS = ["Windows 10", "Windows 11", "macOS", "Android", "iOS"]
IDIOMAS_VALIDOS = ["es-ES", "en-GB", "en-US", "fr-FR", "de-DE", "it-IT",
                   "pt-PT", "nl-NL", "pl-PL", "ca-ES", "sv-SE", "es-MX"]
PAISES_VALIDOS = ["ES", "GB", "FR", "DE", "IT", "PT", "US", "NL",
                  "IE", "BE", "SE", "DK", "PL", "CH", "AT", "MX"]

df_clean = df[
    df["OS del sistema"].isin(OS_VALIDOS)
    & df["Idioma del OS"].isin(IDIOMAS_VALIDOS)
    & df["Pais de origen"].isin(PAISES_VALIDOS)
    & df["edad"].between(18, 85)
].copy()
print(f"Crudas: {len(df):,} -> limpias: {len(df_clean):,} (descartadas: {len(df) - len(df_clean):,})")
df_clean[["OS del sistema", "Idioma del OS", "Pais de origen", "edad"]].head(3)

## 1. Personas por rangos de edad

In [ ]:
bins = [17, 24, 34, 44, 54, 64, 85]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-85"]
df_clean["rango_edad"] = pd.cut(df_clean["edad"], bins=bins, labels=labels, include_lowest=True)

t1 = (df_clean["rango_edad"].value_counts().reindex(labels).rename("personas").to_frame())
t1["%"] = (t1["personas"] / t1["personas"].sum() * 100).round(2)
t1  # Tabla 1: personas por rango de edad

ax = t1["personas"].plot(kind="bar", figsize=(8, 4), title="Personas por rango de edad")
ax.set_xlabel("Rango de edad")
ax.set_ylabel("Personas")
for i, v in enumerate(t1["personas"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 2. SO de PC vs. SO de móvil / tablet

In [ ]:
mapa_tipo = {"Windows 10": "PC", "Windows 11": "PC", "macOS": "PC",
             "Android": "Móvil / tablet", "iOS": "Móvil / tablet"}
df_clean["tipo_so"] = df_clean["OS del sistema"].map(mapa_tipo)

t2 = df_clean["tipo_so"].value_counts().rename("personas").to_frame()
t2["%"] = (t2["personas"] / t2["personas"].sum() * 100).round(2)
t2  # Tabla 2: PC (Windows 10/11, macOS) vs. móvil/tablet (Android, iOS)

ax = t2["personas"].plot(kind="bar", figsize=(6, 4), title="PC vs. móvil / tablet")
ax.set_xlabel("Tipo de SO")
ax.set_ylabel("Personas")
for i, v in enumerate(t2["personas"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

# Detalle por SO, por si se quiere ver el desglose:
detalle_so = df_clean["OS del sistema"].value_counts().rename("personas").to_frame()
detalle_so["%"] = (detalle_so["personas"] / detalle_so["personas"].sum() * 100).round(2)
detalle_so

## 3. Países de origen más frecuentes

In [ ]:
t3 = df_clean["Pais de origen"].value_counts().rename("personas").to_frame()
t3["%"] = (t3["personas"] / t3["personas"].sum() * 100).round(2)
t3  # Tabla 3: países de los que viene más gente (ordenada de mayor a menor)

top = t3.head(10)
ax = top["personas"].plot(kind="barh", figsize=(8, 5), title="Top 10 países de origen")
ax.set_xlabel("Personas")
ax.set_ylabel("País")
ax.invert_yaxis()
for i, v in enumerate(top["personas"]):
    ax.text(v, i, f" {v:,}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Personas por idioma del SO

In [ ]:
t4 = df_clean["Idioma del OS"].value_counts().rename("personas").to_frame()
t4["%"] = (t4["personas"] / t4["personas"].sum() * 100).round(2)
t4  # Tabla 4: personas por idioma

ax = t4["personas"].plot(kind="bar", figsize=(10, 4), title="Personas por idioma del SO")
ax.set_xlabel("Idioma")
ax.set_ylabel("Personas")
plt.xticks(rotation=45, ha="right")
for i, v in enumerate(t4["personas"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8, rotation=0)
plt.tight_layout()
plt.show()

## 5. Ventas — fuente y métrica

Datos: `data/synthetic_sales.csv` (243.485 transacciones de mayo de 2025, generado con `generadores/generador_indices.py`).

Métrica de los tres gráficos: **unidades vendidas** (suma bruta de `items_sold` por grupo).

In [ ]:
candidatos_v = [Path("../data/synthetic_sales.csv"), Path("data/synthetic_sales.csv")]
CSV_VENTAS = next((p for p in candidatos_v if p.exists()), candidatos_v[0])
print(f"CSV ventas: {CSV_VENTAS.resolve()}")

ventas = pd.read_csv(CSV_VENTAS)
ventas["ts"] = pd.to_datetime(ventas["timestamp"])
ventas["hora"] = ventas["ts"].dt.hour
print(f"Transacciones: {len(ventas):,} | Unidades totales: {ventas['items_sold'].sum():,}")
ventas.head(3)

### 5.1 Tipo de tienda que más vende (unidades)

In [ ]:
v_tipo = ventas.groupby("shop_type")["items_sold"].sum().sort_values(ascending=False)
v_tipo  # Unidades por tipo de tienda

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(v_tipo, labels=v_tipo.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Tipo de tienda que más vende (unidades)")
plt.tight_layout()
plt.show()

### 5.2 Tramo de souvenir que más vende (unidades)

In [ ]:
souv = ventas[ventas["shop_type"] == "souvenir"]
v_souv = souv.groupby("price_bracket")["items_sold"].sum().sort_values(ascending=False)
v_souv  # Unidades por tramo: budget / mid-range / premium

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(v_souv, labels=v_souv.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Tramo de souvenir que más vende (unidades)")
plt.tight_layout()
plt.show()

### 5.3 Horas en las que más se compra (unidades)

In [ ]:
v_hora = ventas.groupby("hora")["items_sold"].sum().sort_index()
v_hora  # Unidades por hora del día (0-23)

fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(v_hora, labels=[f"{h:02d}h" for h in v_hora.index], startangle=90,
       autopct=lambda p: f"{p:.0f}%" if p > 4 else "")
ax.set_title("Horas en las que más se compra (unidades)")
plt.tight_layout()
plt.show()